In [7]:
import sqlite3
import pandas as pd
import requests

def fetch_world_bank_data(indicator_code, column_name):
    """Fetches data from the World Bank API for a specific indicator."""
    print(f"Fetching {column_name}...")
    # The API URL fetches data from 2000 to the present for all countries
    url = f"http://api.worldbank.org/v2/country/all/indicator/{indicator_code}?date=2000:2023&format=json&per_page=10000"
    
    response = requests.get(url).json()
    
    # Check if we got valid data back
    if len(response) > 1 and response[1]:
        data = response[1]
        records = [
            {
                "country": item["country"]["value"],
                "country_code": item["countryiso3code"],
                "year": int(item["date"]),
                column_name: item["value"]
            }
            for item in data if item["countryiso3code"] != ""  # Skip non-country regions
        ]
        return pd.DataFrame(records)
    return pd.DataFrame()

# 1. Fetch Indicators
# SP.DYN.LE00.IN = Life Expectancy at Birth
# SH.XPD.CHEX.PP.CD = Current health expenditure per capita, PPP
# SP.POP.TOTL = Total Population
df_life = fetch_world_bank_data("SP.DYN.LE00.IN", "life_expectancy")
df_health = fetch_world_bank_data("SH.XPD.CHEX.PP.CD", "health_expenditure")
df_pop = fetch_world_bank_data("SP.POP.TOTL", "population")

# 2. Merge the DataFrames together
print("Merging datasets...")
df_merged = pd.merge(df_life, df_health, on=["country", "country_code", "year"], how="outer")
df_merged = pd.merge(df_merged, df_pop, on=["country", "country_code", "year"], how="outer")

# 3. Clean up the data (drop rows that have completely empty health/life metrics)
df_clean = df_merged.dropna(subset=["life_expectancy", "health_expenditure"], how="all")

# 4. Save to SQLite Database
print("Saving to local SQLite database...")
conn = sqlite3.connect("../database/healthcare.db")
df_clean.to_sql("health_metrics", conn, if_exists="replace", index=False)

# 5. Verify the data
print("\n--- Data Ingestion Complete ---")
sample_query = "SELECT country, year, life_expectancy, health_expenditure, population FROM health_metrics WHERE health_expenditure IS NOT NULL LIMIT 5;"
print(pd.read_sql_query(sample_query, conn))

conn.close()

Fetching life_expectancy...
Fetching health_expenditure...
Fetching population...
Merging datasets...
Saving to local SQLite database...

--- Data Ingestion Complete ---
       country  year  life_expectancy  health_expenditure  population
0  Afghanistan  2002           56.225           85.857495    21378117
1  Afghanistan  2003           57.171           85.933025    22733049
2  Afghanistan  2004           57.810           93.935804    23560654
3  Afghanistan  2005           58.247          105.927706    24404567
4  Afghanistan  2006           58.553          118.405820    25424094
